# Vector Store Practice Notebook

Use this notebook to practice file management, vector store APIs, semantic search, and file search.

## 1. Setup

Before running:
1. Configure OCI credentials in `sandbox.yaml`.
2. Install dependencies (`uv sync`).
3. Set the IDs and local file path in the next cell.

In [ ]:
import os
from openai import OpenAI
from openai_sdk.openai_client_provider import OpenAIClientProvider

client: OpenAI = OpenAIClientProvider().oci_openai_client
MODEL_ID = 'openai.gpt-5.2'
print('Client ready')

In [ ]:
LOCAL_FILE_PATH = ''
VECTOR_STORE_ID = ''
TARGET_FILE_ID = ''
FILE_IDS_FOR_BATCH = []
DELETE_VECTOR_STORE_AT_END = False

## 2. File Management

In [ ]:
if LOCAL_FILE_PATH:
    with open(LOCAL_FILE_PATH, 'rb') as fh:
        uploaded_file = client.files.create(file=fh, purpose='user_data')
    TARGET_FILE_ID = TARGET_FILE_ID or uploaded_file.id
    print('Uploaded file:', uploaded_file.id)
else:
    print('Skip upload: set LOCAL_FILE_PATH first')

In [ ]:
files_list = client.files.list(order='asc', limit=20)
print('Files returned:', len(files_list.data))

In [ ]:
if TARGET_FILE_ID:
    print(client.files.retrieve(file_id=TARGET_FILE_ID))
    print(client.files.content(file_id=TARGET_FILE_ID))
else:
    print('Skip retrieve/content: set TARGET_FILE_ID')

## 3. Vector Store Lifecycle

In [ ]:
if not VECTOR_STORE_ID:
    created_store = client.vector_stores.create(
        name='workshop-vector-store',
        description='Created from vector_store.ipynb',
        metadata={'topic': 'workshop', 'source': 'notebook'},
    )
    VECTOR_STORE_ID = created_store.id
    print('Created vector store:', VECTOR_STORE_ID)
else:
    print('Using existing vector store:', VECTOR_STORE_ID)

In [ ]:
store_obj = client.vector_stores.retrieve(vector_store_id=VECTOR_STORE_ID)
store_obj

In [ ]:
updated_store = client.vector_stores.update(
    vector_store_id=VECTOR_STORE_ID,
    metadata={'topic': 'workshop', 'state': 'updated'},
)
updated_store

## 4. Semantic Search + Responses File Search

In [ ]:
semantic_result = client.vector_stores.search(
    vector_store_id=VECTOR_STORE_ID,
    query='Summarize the most relevant points from this store.',
    max_num_results=8,
)
semantic_result

In [ ]:
file_search_response = client.responses.create(
    model=MODEL_ID,
    input='What does this vector store say about OCI?',
    tools=[{'type': 'file_search', 'vector_store_ids': [VECTOR_STORE_ID]}],
)
file_search_response.output_text

## 5. Optional File Attach + Batch

In [ ]:
if TARGET_FILE_ID:
    attach_result = client.vector_stores.files.create(
        vector_store_id=VECTOR_STORE_ID,
        file_id=TARGET_FILE_ID,
        attributes={'category': 'sample'},
    )
    print('Attached file:', attach_result.id)
else:
    print('Skip attach: set TARGET_FILE_ID')

In [ ]:
batch_file_ids = FILE_IDS_FOR_BATCH or ([TARGET_FILE_ID] if TARGET_FILE_ID else [])
if batch_file_ids:
    batch = client.vector_stores.file_batches.create(
        vector_store_id=VECTOR_STORE_ID,
        file_ids=batch_file_ids,
        attributes={'category': 'sample'},
    )
    print('Created batch:', batch.id)
    print(client.vector_stores.file_batches.retrieve(vector_store_id=VECTOR_STORE_ID, batch_id=batch.id))
else:
    print('Skip batch: provide FILE_IDS_FOR_BATCH or TARGET_FILE_ID')

## 6. Optional Cleanup

In [ ]:
if DELETE_VECTOR_STORE_AT_END and VECTOR_STORE_ID:
    print(client.vector_stores.delete(vector_store_id=VECTOR_STORE_ID))
else:
    print('Skipping vector store delete. Set DELETE_VECTOR_STORE_AT_END=True to clean up.')